# 04 · Uber — Precio dinámico de extremo a extremo

**Problema:** un viernes a las 18:00 en el centro, con lluvia, ¿cuántas solicitudes llegarán y qué multiplicador de precio equilibra demanda y conductores?

Cuatro algoritmos encadenados, cada uno alimenta al siguiente:

1. **Series temporales** — pronosticar la demanda de los próximos 15 minutos.
2. **k-means** — agrupar zonas con patrones parecidos.
3. **Regresión** — estimar cuánto cae la demanda al subir el precio y hallar el precio de equilibrio.
4. **Bucle de realimentación** — ajustar el multiplicador cada 5 minutos con lo que ocurre de verdad.

> Simulador didáctico con números sintéticos. **No es el sistema de precios de Uber.** Reproduce los cálculos de las diapositivas de `uber/es` (componentes `TimeSeriesViz`, `KMeansViz`, `RegressionViz` y `PricingLoopViz`).

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

import innoit_style as inno
from innoit_style import MORADO, CORAL, TEAL, LAVANDA, LAVANDA_OSCURA, GRIS, TEXTO_SUAVE, PALETA, cascada
inno.aplicar()   # colores, tipografía Urbane y mapas de color de InnoIT

## 1 · Series temporales: pronóstico de demanda

1. **Base:** media ponderada de los últimos 4 viernes a las 18:00 (más peso a los recientes).
2. **Lluvia:** se multiplica por el efecto histórico de la lluvia.
3. **En vivo (*nowcast*):** se comparan las solicitudes de los primeros 5 minutos con lo esperado (un tercio del total) y se corrige la mitad de la desviación.

Mueve los controles y mira cómo cambia cada escalón.

In [ ]:
VIERNES = [120, 132, 128, 140]
PESOS = {'Recientes pesan más (0,1 · 0,2 · 0,3 · 0,4)': [.1, .2, .3, .4], 'Media simple (0,25 cada uno)': [.25]*4}

def pronostico(pesos='Recientes pesan más (0,1 · 0,2 · 0,3 · 0,4)', lluvia=1.18, observado_5min=58, viernes_1=140):
    v = VIERNES[:3] + [viernes_1]; w = PESOS[pesos]
    base = sum(a*b for a, b in zip(v, w))
    con_lluvia = base * lluvia
    esperado5 = con_lluvia / 3
    desvio = observado_5min / esperado5 - 1
    nowcast = con_lluvia * (1 + desvio / 2)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw=dict(width_ratios=[.8, 1.2]))
    a1.bar(['Vie -4', 'Vie -3', 'Vie -2', 'Vie -1'], v, color=MORADO, alpha=.8)
    for i, (x, p) in enumerate(zip(v, w)): a1.text(i, 8, f'{x}\npeso {p}', ha='center', va='bottom', fontsize=8.5, color='white', fontweight=600)
    a1.axhline(base, color=CORAL, ls='--', label=f'Media ponderada {base:.1f}')
    a1.set(ylim=(0, 200), ylabel='Solicitudes en 15 min', title='Últimos 4 viernes · 18:00'); a1.legend(loc='upper left')
    cascada(a2, base, ['Lluvia', 'En vivo'], [con_lluvia - base, nowcast - con_lluvia], 'Base', 'Pronóstico', '{:.1f}')
    a2.set(ylim=(0, 220), title='Ajustes del pronóstico')
    plt.tight_layout(); plt.show()
    display(Markdown(
        f'- Base = Σ viernes × peso = **{base:.1f}**\n'
        f'- Con lluvia = {base:.1f} × {lluvia:.2f} = **{con_lluvia:.1f}**\n'
        f'- Esperado en 5 min = {con_lluvia:.1f} / 3 = {esperado5:.1f}; observado {observado_5min} → desvío {desvio:+.1%}\n'
        f'- Pronóstico = {con_lluvia:.1f} × (1 + {desvio:+.3f} / 2) = **{nowcast:.1f} solicitudes en 15 min**'))

widgets.interact(pronostico,
    pesos=widgets.Dropdown(options=list(PESOS), description='Pesos'),
    lluvia=widgets.FloatSlider(value=1.18, min=1.0, max=1.5, step=.01, description='Efecto lluvia'),
    observado_5min=widgets.IntSlider(58, 30, 90, description='Obs. 5 min'),
    viernes_1=widgets.IntSlider(140, 100, 180, description='Vie −1'));

## 2 · k-means: agrupar zonas parecidas

Cada zona se describe por el % de su demanda en la **hora punta de la mañana** y el % **de madrugada**. Zonas con patrones parecidos pueden compartir pronósticos y precios.

k-means alterna dos pasos hasta que nada cambia:
- **Asignar:** cada zona va al centro más cercano.
- **Mover:** cada centro pasa a la media de sus zonas.

Los centros iniciales de la diapositiva están mal colocados a propósito (dos caen en zonas de negocios). Prueba también inicios aleatorios y pulsa ▶.

In [ ]:
ZONAS = {'A': [48, 4], 'B': [55, 6], 'C': [52, 3], 'D': [30, 12], 'E': [26, 16], 'F': [34, 10], 'G': [8, 42], 'H': [12, 48], 'I': [10, 38]}
P = np.array(list(ZONAS.values()), float); NOMBRES_Z = list(ZONAS)
COLS = [TEAL, MORADO, CORAL]

def kmeans_pasos(centros):
    pasos, c = [], np.array(centros, float)
    pasos.append(('Centros iniciales', None, c.copy()))
    prev = None
    for it in range(1, 20):
        asig = np.argmin(((P[:, None] - c[None])**2).sum(-1), 1)
        pasos.append((f'Iteración {it} · asignar', asig.copy(), c.copy()))
        if prev is not None and (asig == prev).all():
            pasos.append((f'Convergido en {it} iteraciones', asig.copy(), c.copy())); break
        c = np.array([P[asig == k].mean(0) if (asig == k).any() else c[k] for k in range(3)])
        pasos.append((f'Iteración {it} · mover centros', asig.copy(), c.copy()))
        prev = asig
    return pasos

def inicio(nombre):
    if nombre == 'Diapositiva (A, B, I)': return P[[0, 1, 8]]
    return P[np.random.default_rng(int(nombre.split()[-1])).choice(9, 3, replace=False)]

def dibujar_km(inicio_centros, paso):
    pasos = kmeans_pasos(inicio(inicio_centros))
    titulo, asig, c = pasos[min(paso, len(pasos) - 1)]
    fig, ax = plt.subplots(figsize=(6.5, 4.6))
    col = [GRIS]*9 if asig is None else [COLS[a] for a in asig]
    ax.scatter(*P.T, s=180, c=col, edgecolor='white', zorder=2)
    for nm, (x, y), a in zip(NOMBRES_Z, P, [None]*9 if asig is None else asig):
        ax.text(x, y, nm, ha='center', va='center', color='white', fontsize=9, fontweight=600, zorder=3)
        if a is not None: ax.plot([x, c[a, 0]], [y, c[a, 1]], color=COLS[a], lw=.7, alpha=.5)
    ax.scatter(*c.T, marker='X', s=480, c=COLS, edgecolor=MORADO, alpha=.45, zorder=1)
    inercia = 0 if asig is None else ((P - c[asig])**2).sum()
    ax.set(xlim=(0, 62), ylim=(0, 55), xlabel='% demanda hora punta mañana', ylabel='% demanda de madrugada',
           title=f'{titulo}' + (f' · inercia {inercia:.0f}' if asig is not None else ''))
    plt.show()

sel = widgets.Dropdown(options=['Diapositiva (A, B, I)'] + [f'Aleatorio semilla {s}' for s in range(1, 9)], description='Inicio')
play = widgets.Play(min=0, max=12, interval=1200)
slider = widgets.IntSlider(min=0, max=12, description='Paso')
widgets.jslink((play, 'value'), (slider, 'value'))
display(sel, widgets.HBox([play, slider]), widgets.interactive_output(dibujar_km, {'inicio_centros': sel, 'paso': slider}))

Resultado final: residencial (D, E, F), negocios (A, B, C) y ocio nocturno (G, H, I). Comprobamos con `scikit-learn`:

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(P)
final = kmeans_pasos(P[[0, 1, 8]])[-1][1]
print(f'Coincidencia (ARI): {adjusted_rand_score(km.labels_, final):.2f} · el número de cada grupo es arbitrario; importa quién va con quién')
pd.DataFrame({'zona': NOMBRES_Z, 'grupo sklearn': km.labels_, 'grupo paso a paso': final})

## 3 · Regresión: elasticidad y precio de equilibrio

Con datos históricos de qué % de la demanda se mantiene a cada multiplicador, ajustamos una curva log-lineal por mínimos cuadrados:

$$\ln(\%) = a - \beta\,(m - 1) \quad\Rightarrow\quad \text{demanda}(m) = D_0 \cdot e^{-\beta (m-1)}$$

La oferta de conductores sube con el precio: $\text{oferta}(m) = S_0\,(1 + s\,(m-1))$. El **multiplicador de equilibrio** $m^*$ es donde se cruzan; lo hallamos por bisección.

In [ ]:
obs_m = np.array([1.0, 1.2, 1.5, 1.8, 2.0, 2.5])
obs_pct = np.array([100, 92, 79, 68, 61, 47])
xs, ys = obs_m - 1, np.log(obs_pct)
pendiente = ((xs - xs.mean()) * (ys - ys.mean())).sum() / ((xs - xs.mean())**2).sum()
BETA = -pendiente
print(f'β ajustado = {BETA:.3f}  →  cada +0,1 en el multiplicador conserva el {np.exp(-BETA*.1):.1%} de la demanda')

def biseccion(D0, S0, s, iters=60):
    demanda = lambda m: D0*np.exp(-BETA*(m - 1)); oferta = lambda m: S0*(1 + s*(m - 1))
    lo, hi, pasos = 1., 4., []
    if demanda(lo) <= oferta(lo): return 1.0, [(lo, lo)]          # sin escasez: precio normal
    for _ in range(iters):
        mid = (lo + hi) / 2; pasos.append((lo, hi))
        if demanda(mid) > oferta(mid): lo = mid
        else: hi = mid
    return (lo + hi) / 2, pasos

def equilibrio(demanda_base=165, conductores=110, sensib_oferta=.6, paso_biseccion=60):
    m_star, pasos = biseccion(demanda_base, conductores, sensib_oferta)
    lo, hi = pasos[min(paso_biseccion, len(pasos)) - 1] if paso_biseccion else (1, 4)
    ms = np.linspace(1, 2.6, 200)
    fig, ax = plt.subplots(figsize=(9, 4.4))
    ax.plot(ms, demanda_base*np.exp(-BETA*(ms - 1)), color=CORAL, lw=2, label='Demanda (curva ajustada)')
    ax.scatter(obs_m, demanda_base*obs_pct/100, color=CORAL, zorder=3, label='Observaciones históricas')
    ax.plot(ms, conductores*(1 + sensib_oferta*(ms - 1)), color=TEAL, lw=2, label='Oferta de conductores')
    ax.axvspan(lo, min(hi, 2.6), color=MORADO, alpha=.08, label=f'Intervalo de bisección (paso {paso_biseccion})')
    v = demanda_base*np.exp(-BETA*(m_star - 1))
    ax.scatter([m_star], [v], s=120, color=MORADO, zorder=4)
    ax.annotate(f'm* = {m_star:.2f}×\n{v:.0f} viajes', (m_star, v), (12, 12), textcoords='offset points')
    ax.set(xlim=(1, 2.6), ylim=(0, 260), xlabel='Multiplicador de precio (m)', ylabel='Solicitudes y conductores en 15 min')
    ax.legend(fontsize=8); plt.show()

widgets.interact(equilibrio,
    demanda_base=widgets.IntSlider(165, 80, 240, description='Demanda D₀'),
    conductores=widgets.IntSlider(110, 50, 200, description='Conductores S₀'),
    sensib_oferta=widgets.FloatSlider(value=.6, min=0, max=1.5, step=.05, description='Sensib. oferta'),
    paso_biseccion=widgets.IntSlider(60, 0, 15, description='Paso bisección'));

La demanda base de 165 viene del pronóstico del apartado 1. Baja **Paso bisección** a 0 y súbelo uno a uno para ver cómo el intervalo se parte por la mitad hasta encerrar $m^*$.

## 4 · Bucle de realimentación cada 5 minutos

El equilibrio es una estimación. En producción el precio se corrige con lo que pasa:

$$m_{t+1} = \text{clip}\big(m_t + \eta\,(R_t - 1),\ 1,\ 3\big), \qquad R_t = \frac{\text{demanda}_t}{\text{conductores}_t}$$

y los conductores se acercan a la mitad de la oferta objetivo en cada tick. Pulsa ▶ para ver los 10 ticks de 18:00 a 18:45, y prueba un $\eta$ alto: el precio empieza a **oscilar**.

In [ ]:
ESCENARIOS = {
    'Pico y vuelta a la calma': [165]*6 + [130, 110, 100, 100],
    'Pico sostenido': [165]*10,
    'Salida de concierto': [120]*3 + [230]*3 + [140]*4,
}

def simular(bases, eta, beta, vel_conductores):
    m, s, filas = 1., 110., []
    for i, b in enumerate(bases):
        d = b*np.exp(-beta*(m - 1)); r = d / s
        filas.append(dict(hora=f'18:{i*5:02d}', demanda_base=b, multiplicador=m, demanda=d, conductores=s, R=r))
        m_new = float(np.clip(m + eta*(r - 1), 1, 3))
        s = s + vel_conductores*(110*(1 + .6*(m_new - 1)) - s); m = m_new
    return pd.DataFrame(filas)

def bucle(escenario, eta, beta, vel_conductores, tick):
    t = simular(ESCENARIOS[escenario], eta, beta, vel_conductores)
    vis = t.iloc[:tick + 1]
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 5.6), sharex=True, gridspec_kw=dict(height_ratios=[1.4, 1]))
    a1.plot(t.hora, t.multiplicador, color=LAVANDA_OSCURA, lw=1)
    a1.plot(vis.hora, vis.multiplicador, 'o-', color=MORADO, lw=2)
    for h, v in zip(vis.hora, vis.multiplicador): a1.text(h, v + .03, f'{v:.2f}×', ha='center', fontsize=8)
    a1.set(ylabel='Multiplicador', ylim=(.9, max(1.6, t.multiplicador.max() + .15)), title=f'Multiplicador aplicado · {escenario}')
    a2.bar(vis.hora, vis.R - 1, bottom=1, color=[CORAL if r > 1 else TEAL for r in vis.R])
    a2.axhline(1, color=MORADO, lw=.8)
    a2.set(ylabel='Demanda / conductores', ylim=(.4, max(1.7, t.R.max() + .1)),
           title='R > 1 faltan conductores (coral) · R < 1 sobran (teal)')
    a2.set_xticks(range(len(t)), t.hora)
    plt.tight_layout(); plt.show()
    display(vis.round(2).set_index('hora'))

controles = dict(
    escenario=widgets.Dropdown(options=list(ESCENARIOS), description='Escenario'),
    eta=widgets.FloatSlider(value=.8, min=.1, max=3, step=.1, description='η (reactividad)'),
    beta=widgets.FloatSlider(value=round(BETA, 3), min=.1, max=1.5, step=.01, description='β (elasticidad)'),
    vel_conductores=widgets.FloatSlider(value=.5, min=.1, max=1, step=.05, description='Vel. conductores'),
    tick=widgets.IntSlider(9, 0, 9, description='Tick'))
play = widgets.Play(min=0, max=9, interval=900)
widgets.jslink((play, 'value'), (controles['tick'], 'value'))
display(widgets.VBox([controles['escenario'], widgets.HBox([controles['eta'], controles['beta']]),
                      controles['vel_conductores'], widgets.HBox([play, controles['tick']])]),
        widgets.interactive_output(bucle, controles))

## Para llevarse

- Ningún algoritmo resuelve el problema solo: **pronóstico → segmentación → elasticidad → control**.
- La regresión da un punto de partida razonable; el bucle corrige los errores del modelo con datos en vivo.
- Un controlador demasiado reactivo (η alto) oscila: en precios, **la estabilidad es parte del diseño**.